In [10]:
!pip install arch
!pip install pytorch-lightning
import pytorch_lightning as pl
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np


class TimeseriesDataset(Dataset):
    '''
    Custom Dataset subclass.
    Serves as input to DataLoader to transform X
      into sequence data using rolling window.
    DataLoader using this dataset will output batches
      of `(batch_size, seq_len, n_features)` shape.
    Suitable as an input to RNNs.
    '''

    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int = 1):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).float()
        self.seq_len = seq_len

    def __len__(self):
        return self.X.__len__() - (self.seq_len)

    def __getitem__(self, index):
        return (self.X[index:index + self.seq_len], self.y[index + self.seq_len - 1])


class ValueAtRiskDataModule(pl.LightningDataModule):
    '''
    PyTorch Lighting DataModule subclass:
    https://pytorch-lightning.readthedocs.io/en/latest/datamodules.html

    Serves the purpose of aggregating all data loading
      and processing work in one place.
    '''

    def __init__(self, df, training_length=1000, seq_len=1, batch_size=128, num_workers=0):
        super().__init__()
        self.df = df
        self.test_case = 0
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.X_train = None
        self.y_train = None
        self.X_val = None
        self.y_val = None
        self.X_test = None
        self.X_test = None
        self.preprocessing = None
        self.training_length = training_length

    def prepare_data(self):
        pass

    def move_timestep(self):
        self.test_case += 1

    def setup_train(self):
        data = self.df.iloc[self.test_case:self.training_length + self.test_case]

        X = data[['log_returns']].values[:self.training_length]
        y = data[['log_returns']].shift(-1).values[:self.training_length]

        self.preprocessing = StandardScaler()
        self.preprocessing.fit(X)

        self.X_train = self.preprocessing.transform(X)
        self.y_train = self.preprocessing.transform(y)

        self.X_test = data[['log_returns']].values[-self.seq_len - 1:-1]
        self.X_test = self.preprocessing.transform(self.X_test)

        self.X_test = torch.tensor(self.X_test, dtype=torch.float).unsqueeze(0)

    def train_dataloader(self):
        train_dataset = TimeseriesDataset(self.X_train,
                                          self.y_train,
                                          seq_len=self.seq_len)
        train_loader = DataLoader(train_dataset,
                                  batch_size=self.batch_size,
                                  shuffle=False,
                                  num_workers=self.num_workers)

        return train_loader

    def gather_prediction(self, prediction):
        self.df.loc[self.df.index[self.training_length + self.test_case], 'VaR'] = prediction
        print("Date: {0} -> RR: {1} | VaR: {2}".format(*(self.df.index[self.training_length + self.test_case],) + tuple(
            self.df.loc[self.df.index[self.training_length + self.test_case], ['log_returns', 'VaR']])))

In [11]:
import torch
import numpy as np


def caviar_loss(true, pred, pval=0.025):
    # return torch.mean(-1*((true < var).float() - pval) * (true - var))
    return torch.mean(torch.max(pval*(true - pred), (pval-1)*(true - pred)))


def huber_loss(true, var, pval=torch.tensor(0.025), eps=torch.tensor(0.025)):
    x = true - var
    return torch.mean(torch.cat([
        x[x <= (pval - 1) * eps] * (pval - 1) - 1 / 2 * (pval - 1) ** 2 * eps,
        x[(x > (pval - 1) * eps) & (x <= pval * eps)] ** 2 / (2 * eps),
        x[x > pval * eps] * pval - 1 / 2 * pval ** 2 * eps
    ]))


def garch_normal_loss(true, vol):
    return 1 / 2 * torch.mean(torch.log(vol) + true ** 2 / vol)  # + tf.math.log(2 * tf.constant(np.pi))


def student_loss(true, pred):
    vol = pred[0]
    df = pred[1] + 2

    llh = + 1/2 * (
        torch.log(vol) + (1+df)*torch.log(1 + torch.square(true)/(vol * (df - 2)))
    )

    return llh


def hansen_garch_skewed_student_loss(true, pred):

    vol = pred[:, 0]
    df = pred[:, 2]
    skewness = pred[:, 1]
    #TODO repair
    true = true[:, 0]

    c = torch.lgamma((df + 1)/2) - torch.lgamma(df / 2) - torch.log(torch.pi * (df - 2)) / 2

    a = 4 * skewness * torch.exp(c) * (df-2) / (df-1)

    b = torch.sqrt(1 + 3*torch.square(skewness) - torch.square(a))

    z = true / torch.sqrt(vol)

    lls = torch.log(b) + c - torch.log(vol) / 2

    llf_resid = torch.square((b * z + a) / (1 + torch.sign(z + a / b) * skewness))

    lls -= (df + 1) / 2 * torch.log(1 + llf_resid / (df - 2))

    lls *= -1

    return torch.mean(lls)

In [12]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
import numpy as np
from arch.univariate import SkewStudent


class VaRNet(pl.LightningModule):

    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.criterion = criterion
        self.learning_rate = learning_rate
        self.dist = dist

        self.lstm = nn.LSTM(input_size=n_features,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout,
                            batch_first=True)
        self.linear = nn.Linear(hidden_size, 64)
        self.linear2 = torch.nn.Linear(64, 32)
        self.linear3 = torch.nn.Linear(32, 1)

    def forward(self, x):
        # lstm_out = (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(x)
        y_pred = self.linear(lstm_out[:, -1])
        y_pred = self.linear2(y_pred)
        y_pred = self.linear3(y_pred)
        return y_pred

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y, y_hat)
        self.log('train_loss', loss, on_epoch=True)
        return loss

    def predict_var(self, x):
        if self.dist is None:
            return self.forward(x).detach().numpy()
        elif isinstance(self.dist(), SkewStudent):
            var = self.forward(x).detach().numpy()[0]
            return np.array([np.sqrt(var[:1]) * self.dist(eta=var[2], lam=var[1]).ppf(0.025)])
        else:
            return np.sqrt(self.forward(x).detach().numpy()) * self.dist().ppf(0.025)


class GARCHVaRNet(VaRNet):

    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__(n_features,
                         hidden_size,
                         seq_len,
                         batch_size,
                         num_layers,
                         dropout,
                         learning_rate,
                         criterion,
                         dist)

        self.softplus = torch.nn.Softplus()

    def forward(self, x):
        # lstm_out = (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(x)
        y_pred = self.linear(lstm_out[:, -1])
        y_pred = self.linear2(y_pred)
        y_pred = self.linear3(y_pred)
        return self.softplus(y_pred)

class SkewedGARCHVaRNet(GARCHVaRNet):

    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__(n_features,
                         hidden_size,
                         seq_len,
                         batch_size,
                         num_layers,
                         dropout,
                         learning_rate,
                         criterion,
                         dist)

        self.linear3_1 = torch.nn.Linear(32, 1)
        self.linear3_2 = torch.nn.Linear(32, 1)
        self.linear3_3 = torch.nn.Linear(32, 1)

        self.tanh = torch.nn.Tanh()
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        # lstm_out = (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(x)
        y_pred = self.linear(lstm_out[:, -1])
        y_pred = self.linear2(y_pred)

        y_pred_1 = self.softplus(self.linear3_1(y_pred))  # vol
        y_pred_2 = self.tanh(self.linear3_2(y_pred))  # skew
        y_pred_3 = self.relu(self.linear3_3(y_pred)) + 2.05  # df

        return torch.cat([
            y_pred_1,
            y_pred_2,
            y_pred_3
        ], 1)

In [14]:
import pandas as pd
import argparse
from pytorch_lightning import Trainer, seed_everything, Callback
from pytorch_lightning.loggers import CSVLogger
from scipy.stats import norm

import matplotlib.pyplot as plt


def experiment(model_name):

    if model_name == 'garch_skew':
        model_class = SkewedGARCHVaRNet
        dist = SkewStudent
        loss = hansen_garch_skewed_student_loss
    elif model_name == 'garch_norm':
        model_class = GARCHVaRNet
        dist = norm
        loss = garch_normal_loss
    elif model_name == 'caviar':
        model_class = VaRNet
        dist = None
        loss = caviar_loss
    elif model_name == 'caviar_huber':
        model_class = VaRNet
        dist = None
        loss = huber_loss
    else:
        return


    sample_starts = [
        '2005-01-01',
        '2007-01-01',
        '2013-01-01',
        '2016-01-01'
    ]

    indexes = [
        'wig',
        # 'spx',
        # 'lse'
    ]

    memory_sizes = [
        5,
        # 10,
        # 20,
        # 100
    ]
    seed_everything(1)

    for mem_size in memory_sizes:
        p = dict(
            training_length=1000,
            seq_len=mem_size,
            batch_size=512,
            criterion=loss,
            max_epochs=300,
            n_features=1,
            hidden_size=100,
            num_layers=1,
            dropout=0,
            learning_rate=3e-4,
            num_train=250
        )


        for sample_start in sample_starts:
            path = './data/wig.csv'
            data = pd.read_csv(
                path,
                sep=',',
                index_col='Data'
            )
            data['log_returns'] = data['Zamkniecie'].rolling(2).apply(lambda x: np.log(x[1] / x[0]), raw=True)
            data = data.loc[(data.index > sample_start)]
            data['VaR'] = np.nan

            dm = ValueAtRiskDataModule(
                df=data,
                training_length=p['training_length'],
                seq_len=p['seq_len'],
                batch_size=p['batch_size'],
            )

            for test_case in range(p['num_train']):
                csv_logger = CSVLogger('./runs/', name='{}_{}_{}'.format(model_name, mem_size, sample_start), version=str(test_case))
                trainer = Trainer(
                    max_epochs=p['max_epochs'],
                    logger=csv_logger,
                    gpus=1,
                    progress_bar_refresh_rate=20,
                    weights_summary=None
                )

                model = model_class(
                    n_features=p['n_features'],
                    hidden_size=p['hidden_size'],
                    seq_len=p['seq_len'],
                    batch_size=p['batch_size'],
                    criterion=p['criterion'],
                    num_layers=p['num_layers'],
                    dropout=p['dropout'],
                    learning_rate=p['learning_rate'],
                    dist=dist
                )

                dm.setup_train()
                trainer.fit(model, datamodule=dm)
                # model.eval()
                # train_test = TimeseriesDataset(dm.X_train, dm.y_train, 10)
                # pred_out = []
                # true_out = []
                # for i in range(len(train_test)):
                #     pred_out.append(model.forward(torch.tensor(train_test[i][0], dtype=torch.float).unsqueeze(0)).data)
                #     true_out.append(train_test[i][1].data)
                #
                # plt.plot(pred_out)
                # plt.plot(true_out)
                # plt.show()

                # train_test = TimeseriesDataset(dm.X_train, dm.y_train, 100)
                # pred_out = []
                # true_out = []
                # for i in range(len(train_test)):
                #     VaR = model.forward(torch.tensor(train_test[i][0], dtype=torch.float).unsqueeze(0)).detach().numpy()[0]
                #     # dist = skewstudent.skewstudent.SkewStudent(eta=VaR[2], lam=VaR[1])
                #     # var = np.sqrt(VaR[0]) * dist.ppf(0.025)
                #     var = np.sqrt(VaR[0]) * norm().ppf(0.025)
                #     pred_out.append(var)
                #     true_out.append(train_test[i][1].data)
                #
                # plt.plot(pred_out)
                # plt.plot(true_out)
                # plt.show()

                dm.gather_prediction(dm.preprocessing.inverse_transform(model.predict_var(dm.X_test))[0][0])
                dm.move_timestep()

            dm.df[['log_returns', 'VaR']].to_csv('{}_{}_{}.csv'.format(model_name, sample_start, mem_size))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name', metavar='m', type=str,
                        help='model name')
    args = parser.parse_args()
    experiment(args.model_name)


usage: colab_kernel_launcher.py [-h] [--model_name m]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-dcc87dc5-2221-4c94-90f8-65ee9274adc3.json


SystemExit: 2